In [22]:
import requests
import json

# --- CONFIGURATION ---
BASE_URL = "https://des-oracle.bombaykettle.org"

# These are your Service Token credentials
CF_CLIENT_ID = '<client_id>.access'
CF_CLIENT_SECRET = '<client_secret_string>'

HEADERS = {
   'CF-Access-Client-Id': CF_CLIENT_ID,
   'CF-Access-Client-Secret': CF_CLIENT_SECRET,
   'Content-Type': 'application/json'
}


In [23]:
def query_oracle(roll_number: str, pt_hex: str) -> str | None:
    """
    Queries the specific oracle for the given roll_number.
    Returns ciphertext as uppercase hex string or None on error.
    """
    url = f"{BASE_URL}/{roll_number}/api/encrypt"
    payload = {'plaintext': pt_hex}

    print(f"    Querying {roll_number}: 0x{pt_hex}")
    
    try:
        response = requests.post(url, headers=HEADERS, json=payload, timeout=15)
        response.raise_for_status()
        
        result = response.json()
        ciphertext = result.get('ciphertext', '').upper()
        
        # Validate ciphertext format
        if len(ciphertext) == 16 and all(c in '0123456789ABCDEF' for c in ciphertext):
            return ciphertext
        
        # Handle API errors
        if 'detail' in result:
            print(f"    ✗ Oracle error: {result['detail']}")
        else:
            print(f"    ✗ Invalid response: {result}")
        return None
        
    except requests.exceptions.HTTPError as e:
        error_msg = response.json().get('detail', str(e)) if response.text else str(e)
        print(f"    ✗ HTTP {response.status_code}: {error_msg}")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"    ✗ Connection error: {e}")
        return None
        
    except (ValueError, KeyError) as e:
        print(f"    ✗ Invalid JSON response: {e}")
        return None


In [24]:
test_roll_number = "cs24mtech12021"
test_pt = "FEDCBA9876543412" # plaintext

print(f"\n--- Testing Oracle API Connection for {test_roll_number} ---")
test_ct = query_oracle(test_roll_number, test_pt)

if test_ct:
    print(f"Connection test successful! Received: 0x{test_ct}")
else:
    print(f"Connection test FAILED. Check URL, Cloudflare Headers, and Oracle status.")
print(f"--- End Test ---")


--- Testing Oracle API Connection for cs24mtech12021 ---
    Querying cs24mtech12021: 0xFEDCBA9876543412
Connection test successful! Received: 0xAFDFFC4D02D83ECE
--- End Test ---
